# Mutual Fund Analytics - Exploratory Data Analysis

## Data Loading & Setup

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
import os

# Load Real Processed Datasets
nav_history = pd.read_csv('../data/processed/02_nav_history.csv')
fund_master = pd.read_csv('../data/processed/01_fund_master.csv')
transactions = pd.read_csv('../data/processed/08_investor_transactions.csv')
scheme_perf = pd.read_csv('../data/processed/07_scheme_performance.csv')
aum_data = pd.read_csv('../data/processed/03_aum_by_fund_house.csv')
sip_inflows = pd.read_csv('../data/processed/04_monthly_sip_inflows.csv')
category_inflows = pd.read_csv('../data/processed/05_category_inflows.csv')
folio_count = pd.read_csv('../data/processed/06_industry_folio_count.csv')
holdings = pd.read_csv('../data/processed/09_portfolio_holdings.csv')

# Configure Plotting Style
plt.style.use('default')
sns.set_theme(style="whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)

print("Real data loaded successfully for EDA!")


## 1. NAV Trends (Interactive)

In [ ]:

# Merge nav with fund master for names
nav_merged = pd.merge(nav_history, fund_master[['amfi_code', 'scheme_name']], on='amfi_code')
top_5_schemes = nav_merged['amfi_code'].unique()[:5]
plot_data = nav_merged[nav_merged['amfi_code'].isin(top_5_schemes)]

fig = px.line(plot_data, x='date', y='nav', color='scheme_name', title='Daily NAV Trend for Top 5 Schemes')
fig.show()


## 2. Returns Distribution

In [ ]:

plt.figure(figsize=(10,6))
sns.histplot(scheme_perf['return_1yr_pct'].dropna(), bins=15, kde=True, color='blue', label='1Y Return')
plt.title('Distribution of 1-Year Returns')
plt.xlabel('Return (%)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/return_distribution.png', dpi=300)
plt.show()


## 3. NAV Correlation Matrix

In [ ]:

nav_pivot = nav_merged[nav_merged['amfi_code'].isin(top_5_schemes)].pivot(index='date', columns='scheme_name', values='nav')
daily_returns = nav_pivot.pct_change().dropna()
corr = daily_returns.corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Daily Return Correlation Matrix (Top 5 Schemes)')
plt.tight_layout()
plt.savefig('../reports/correlation_matrix.png', dpi=300)
plt.show()


## 4. Risk vs Return Profile

In [ ]:

fig = px.scatter(scheme_perf, x='std_dev_ann_pct', y='return_1yr_pct', color='category', hover_name='scheme_name',
                 title='Risk vs Return Profile (1-Year)')
fig.show()


## 5. AUM by Fund House

In [ ]:

plt.figure(figsize=(12,6))
sns.barplot(data=aum_data.head(10), x='aum_crore', y='fund_house', palette='viridis')
plt.title('Top 10 Fund Houses by Total AUM')
plt.xlabel('AUM (in Crores)')
plt.ylabel('Fund House')
plt.tight_layout()
plt.savefig('../reports/aum_by_fund_house.png', dpi=300)
plt.show()


## 6. Category Inflows

In [ ]:

plt.figure(figsize=(10,6))
sns.barplot(data=category_inflows, x='net_inflow_crore', y='category', palette='magma')
plt.title('Net Inflows by Category')
plt.xlabel('Net Inflow (Crores)')
plt.tight_layout()
plt.savefig('../reports/category_inflow.png', dpi=300)
plt.show()


## 7. Monthly SIP Inflows

In [ ]:

fig = px.line(sip_inflows, x='month', y='sip_amount_crore', title='Monthly SIP Inflows (in Crores)', markers=True)
fig.show()


## 8. Transaction Types Count

In [ ]:

plt.figure(figsize=(8,6))
sns.countplot(data=transactions, x='transaction_type', palette='Set2')
plt.title('Transaction Types Count')
plt.ylabel('Number of Transactions')
plt.tight_layout()
plt.savefig('../reports/transaction_types.png', dpi=300)
plt.show()


## 9. SIP vs Lumpsum Value

In [ ]:

plt.figure(figsize=(8,6))
tx_val = transactions.groupby('transaction_type')['amount_inr'].sum().reset_index()
sns.barplot(data=tx_val, x='transaction_type', y='amount_inr', palette='pastel')
plt.title('Total Investment Value by Type')
plt.ylabel('Amount (INR)')
plt.tight_layout()
plt.savefig('../reports/sip_vs_lumpsum_value.png', dpi=300)
plt.show()


## 10. Investor Age Distribution

In [ ]:

plt.figure(figsize=(10,6))
age_counts = transactions['age_group'].value_counts()
plt.pie(age_counts, labels=age_counts.index, autopct='%1.1f%%', colors=plt.cm.Set3.colors)
plt.title('Investor Age Group Distribution')
plt.tight_layout()
plt.savefig('../reports/age_distribution.png', dpi=300)
plt.show()


## 11. Investor Gender Split

In [ ]:

plt.figure(figsize=(8,6))
gender_counts = transactions['gender'].value_counts()
plt.pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', colors=['skyblue', 'lightpink'])
plt.title('Investor Gender Split')
plt.tight_layout()
plt.savefig('../reports/gender_split.png', dpi=300)
plt.show()


## 12. Top 10 States by Transaction Value

In [ ]:

plt.figure(figsize=(12,6))
state_tx = transactions.groupby('state')['amount_inr'].sum().sort_values(ascending=False).head(10).reset_index()
sns.barplot(data=state_tx, x='amount_inr', y='state', palette='Blues_r')
plt.title('Top 10 States by Investment Value')
plt.xlabel('Amount (INR)')
plt.tight_layout()
plt.savefig('../reports/transactions_by_state.png', dpi=300)
plt.show()


## 13. City Tier Distribution

In [ ]:

plt.figure(figsize=(8,6))
tier_counts = transactions['city_tier'].value_counts()
plt.pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%', colors=['#ff9999','#66b3ff'])
plt.title('City Tier Distribution (T30 vs B30)')
plt.tight_layout()
plt.savefig('../reports/city_tier.png', dpi=300)
plt.show()


## 14. Industry Folio Growth

In [ ]:

plt.figure(figsize=(10,6))
sns.lineplot(data=folio_count, x='year', y='total_folios_lakh', marker='o', linewidth=2, color='green')
plt.title('Industry Folio Growth')
plt.xlabel('Year')
plt.ylabel('Total Folios (Lakhs)')
plt.tight_layout()
plt.savefig('../reports/folio_growth.png', dpi=300)
plt.show()


## 15. Sector Allocation (HDFC Top 100 Example)

In [ ]:

plt.figure(figsize=(10,6))
sector_weights = holdings.groupby('sector')['weight_pct'].sum().sort_values(ascending=False).head(8)
plt.pie(sector_weights, labels=sector_weights.index, autopct='%1.1f%%', cmap='Pastel1')
plt.title('Sector Allocation (Top 8 Sectors)')
plt.tight_layout()
plt.savefig('../reports/sector_allocation.png', dpi=300)
plt.show()


## 16. Top 5 Performing Funds (1-Year Return)

In [ ]:

plt.figure(figsize=(10,6))
top_5_perf = scheme_perf.sort_values('return_1yr_pct', ascending=False).head(5)
sns.barplot(data=top_5_perf, x='return_1yr_pct', y='scheme_name', palette='Greens_r')
plt.title('Top 5 Performing Funds (1-Year Return %)')
plt.xlabel('Return (%)')
plt.tight_layout()
plt.savefig('../reports/top_5_perf.png', dpi=300)
plt.show()
